# 04 — Error Analysis & Grad-CAM
**Project:** Plant Disease Classification  
**Model:** ResNet50 Fine-tuned (Test Acc: 99.66%, Macro-F1: 99.52%)  
**Goal:** Understand where the model fails (Phase 7) and what it looks at (Phase 8 — Grad-CAM).  
**No retraining** — loads saved checkpoint and runs inference only.

---
## 1 — Paths, Imports, Config

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import torchvision.transforms as T
import torchvision.models as models

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"
OUT_DIR       = PROJECT_ROOT / "results" / "error_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Config ─────────────────────────────────────────────────────────────────
BATCH_SIZE   = 32
NUM_WORKERS  = 0
IMG_SIZE     = 224
SEED         = 42
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
np.random.seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"Device  : {DEVICE}")
print(f"Out dir : {OUT_DIR}")

---
## 2 — Load Model & Data

In [ ]:
# ── Load class names from train split ──────────────────────────────────────
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
test_df  = pd.read_csv(PROCESSED_DIR / "test.csv")

num_classes = train_df["encoded_label"].nunique()
class_pairs = (train_df[["encoded_label", "label"]]
               .drop_duplicates()
               .sort_values("encoded_label"))
idx_to_class = dict(zip(class_pairs["encoded_label"], class_pairs["label"]))
class_names  = [idx_to_class[i] for i in range(num_classes)]

print(f"Classes : {num_classes}")
print(f"Test    : {len(test_df):,} images")

In [ ]:
# ── Build & load ResNet50 ───────────────────────────────────────────────────
def build_resnet50(num_classes):
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_resnet50(num_classes).to(DEVICE)
ckpt  = MODELS_DIR / "resnet50_best.pth"
model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
model.eval()
print(f"Loaded checkpoint: {ckpt.name}")
print(f"Parameters       : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Dataset & transforms ────────────────────────────────────────────────────
val_tfms = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

class PlantDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths     = df["image_path"].values
        self.labels    = df["encoded_label"].values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = Path(self.paths[idx])
        if not p.is_absolute():
            p = PROJECT_ROOT / p
        img = Image.open(p).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx], str(self.paths[idx])

test_dataset = PlantDataset(test_df, transform=val_tfms)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f"Test loader: {len(test_loader)} batches")

---
## 3 — Full Test Inference

In [ ]:
all_true, all_pred, all_conf, all_paths = [], [], [], []

with torch.no_grad():
    for images, labels, paths in tqdm(test_loader, desc="Inference"):
        images = images.to(DEVICE)
        with autocast(device_type='cuda'):
            logits = model(images)
        probs      = F.softmax(logits.float(), dim=1)
        confs, preds = probs.max(dim=1)

        all_true.extend(labels.numpy())
        all_pred.extend(preds.cpu().numpy())
        all_conf.extend(confs.cpu().numpy())
        all_paths.extend(paths)

# Build results DataFrame
results_df = pd.DataFrame({
    "image_path" : all_paths,
    "true_idx"   : all_true,
    "pred_idx"   : all_pred,
    "confidence" : all_conf,
})
results_df["true_label"] = results_df["true_idx"].map(idx_to_class)
results_df["pred_label"] = results_df["pred_idx"].map(idx_to_class)
results_df["correct"]    = results_df["true_idx"] == results_df["pred_idx"]

results_df.to_csv(OUT_DIR / "inference_results.csv", index=False)

n_total   = len(results_df)
n_correct = results_df["correct"].sum()
n_wrong   = n_total - n_correct
print(f"Total   : {n_total:,}")
print(f"Correct : {n_correct:,}  ({100*n_correct/n_total:.2f}%)")
print(f"Wrong   : {n_wrong:,}  ({100*n_wrong/n_total:.2f}%)")

---
## 4 — Full Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm_arr = confusion_matrix(results_df["true_idx"], results_df["pred_idx"])

# Short labels for readability
short_names = [n.split("___")[-1].replace("_", " ")[:28] for n in class_names]

fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(cm_arr, ax=ax, cmap="Blues",
            xticklabels=short_names, yticklabels=short_names,
            linewidths=0.3, linecolor="#e0e0e0")
ax.set_title("Confusion Matrix — ResNet50 Fine-tuned (Test Set)", fontsize=14, pad=12)
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("Actual",    fontsize=11)
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_test.png", dpi=200)
plt.show()
print(f"Saved → {OUT_DIR / 'confusion_matrix_test.png'}")

---
## 5 — Top Confused Pairs

In [ ]:
# Extract off-diagonal entries
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm_arr[i, j] > 0:
            confused_pairs.append({
                "true_class" : class_names[i],
                "pred_class" : class_names[j],
                "count"      : int(cm_arr[i, j])
            })

pairs_df = (pd.DataFrame(confused_pairs)
              .sort_values("count", ascending=False)
              .reset_index(drop=True))

top_n = min(10, len(pairs_df))
top_pairs = pairs_df.head(top_n).copy()
top_pairs["pair_label"] = (top_pairs["true_class"].str.split("___").str[-1].str[:22]
                           + " →\n"
                           + top_pairs["pred_class"].str.split("___").str[-1].str[:22])

print(f"Total unique confused pairs : {len(pairs_df)}")
print(f"Total misclassifications    : {pairs_df['count'].sum()}")
print()
print(top_pairs[["true_class", "pred_class", "count"]].to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, max(4, top_n * 0.6)))
bars = ax.barh(top_pairs["pair_label"][::-1], top_pairs["count"][::-1],
               color="#d45f5f", alpha=0.85)
ax.set_xlabel("Misclassification Count")
ax.set_title(f"Top-{top_n} Confused Class Pairs — ResNet50")
for bar in bars:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va="center", fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / "top_confused_pairs.png", dpi=200)
plt.show()
print(f"Saved → {OUT_DIR / 'top_confused_pairs.png'}")

---
## 6 — Misclassification Gallery

In [ ]:
wrong_df = results_df[results_df["correct"] == False].reset_index(drop=True)
print(f"Total misclassified: {len(wrong_df)}")

if len(wrong_df) == 0:
    print("No misclassifications found — perfect test accuracy!")
else:
    n_show = min(len(wrong_df), 16)
    ncols  = min(n_show, 4)
    nrows  = (n_show + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.8))
    axes = np.array(axes).reshape(-1)

    for i in range(n_show):
        row  = wrong_df.iloc[i]
        p    = Path(row["image_path"])
        if not p.is_absolute():
            p = PROJECT_ROOT / p
        img  = Image.open(p).convert("RGB").resize((224, 224))

        true_short = row["true_label"].split("___")[-1].replace("_", " ")
        pred_short = row["pred_label"].split("___")[-1].replace("_", " ")

        axes[i].imshow(img)
        axes[i].set_title(
            f"TRUE: {true_short[:22]}\nPRED: {pred_short[:22]}\nConf: {row['confidence']:.3f}",
            fontsize=7, color="darkred"
        )
        axes[i].axis("off")

    # Hide unused axes
    for j in range(n_show, len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"Misclassification Gallery — ResNet50 ({n_show} of {len(wrong_df)} shown)",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "misclassification_gallery.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved → {OUT_DIR / 'misclassification_gallery.png'}")

---
## 7 — Grad-CAM Implementation
Pure PyTorch hooks — no external libraries, no cv2.
Target: `model.layer4[-1]` (7×7 spatial feature map, highest-level before global avg pool).

In [ ]:
class GradCAM:
    """Grad-CAM for ResNet50 using model.layer4[-1] as the target layer."""

    def __init__(self, model):
        self.model      = model
        self.activation = None
        self.gradient   = None
        self._hooks     = []

    def _register_hooks(self):
        def fwd_hook(module, input, output):
            self.activation = output.detach()

        def bwd_hook(module, grad_input, grad_output):
            self.gradient = grad_output[0].detach()

        target_layer = self.model.layer4[-1]
        self._hooks.append(target_layer.register_forward_hook(fwd_hook))
        self._hooks.append(target_layer.register_full_backward_hook(bwd_hook))

    def _remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def compute(self, img_tensor, class_idx=None):
        """
        img_tensor : (1, 3, 224, 224) on DEVICE, no torch.no_grad()
        class_idx  : int or None (uses predicted class if None)
        Returns    : (224, 224) numpy array, values in [0, 1]
        """
        self._register_hooks()
        try:
            self.model.zero_grad()
            logits = self.model(img_tensor)          # forward — keeps graph
            if class_idx is None:
                class_idx = int(logits.argmax(dim=1).item())

            score = logits[0, class_idx]
            score.backward()                         # backward on class score

            # α_k = mean of gradients over spatial dims → (C,)
            weights = self.gradient.mean(dim=(2, 3), keepdim=True)   # (1, C, 1, 1)

            # Weighted combination of activation maps
            cam = (weights * self.activation).sum(dim=1, keepdim=True)  # (1, 1, 7, 7)
            cam = F.relu(cam)

            # Normalize to [0, 1]
            cam = cam.squeeze().cpu().numpy()
            cam_min, cam_max = cam.min(), cam.max()
            if cam_max - cam_min > 1e-8:
                cam = (cam - cam_min) / (cam_max - cam_min)
            else:
                cam = np.zeros_like(cam)

            # Resize to 224×224
            cam_pil = Image.fromarray((cam * 255).astype(np.uint8))
            cam_pil = cam_pil.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            cam_np  = np.array(cam_pil) / 255.0

            return cam_np, class_idx

        finally:
            self._remove_hooks()
            self.model.zero_grad()


def make_overlay(original_img_pil, cam_np, alpha=0.5):
    """
    Blend original image with jet-colormap heatmap.
    original_img_pil : PIL Image (any size, will be resized to 224)
    cam_np           : (224, 224) float array in [0, 1]
    Returns          : (224, 224, 3) numpy array in [0, 1]
    """
    img_resized = original_img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np      = np.array(img_resized) / 255.0                # (224,224,3)
    heatmap     = cm.jet(cam_np)[:, :, :3]                     # drop alpha → (224,224,3)
    overlay     = alpha * img_np + (1 - alpha) * heatmap
    return np.clip(overlay, 0, 1)


def load_raw_image(rel_path_str):
    """Load PIL image from relative or absolute path."""
    p = Path(rel_path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    return Image.open(p).convert("RGB")


def img_to_tensor(pil_img):
    """Apply val transforms and add batch dim, send to DEVICE."""
    return val_tfms(pil_img).unsqueeze(0).to(DEVICE)


gradcam = GradCAM(model)
print("GradCAM ready — target: model.layer4[-1]")

---
## 8 — Grad-CAM Gallery: Correct High-Confidence (8 images)

In [ ]:
correct_df   = results_df[results_df["correct"] == True].copy()
high_conf_df = correct_df.sort_values("confidence", ascending=False).head(8).reset_index(drop=True)

n_show = len(high_conf_df)
fig, axes = plt.subplots(n_show, 2, figsize=(7, n_show * 3.2))

for i, row in high_conf_df.iterrows():
    pil_img  = load_raw_image(row["image_path"])
    tensor   = img_to_tensor(pil_img)
    cam_np, _ = gradcam.compute(tensor, class_idx=int(row["pred_idx"]))
    overlay  = make_overlay(pil_img, cam_np)

    label_short = row["true_label"].split("___")[-1].replace("_", " ")

    axes[i, 0].imshow(pil_img.resize((IMG_SIZE, IMG_SIZE)))
    axes[i, 0].set_title(f"{label_short[:30]}\nConf: {row['confidence']:.4f}", fontsize=7)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(overlay)
    axes[i, 1].set_title("Grad-CAM", fontsize=7)
    axes[i, 1].axis("off")

    torch.cuda.empty_cache()

fig.suptitle("Grad-CAM — Correct High-Confidence Predictions (ResNet50)",
             fontsize=11, y=1.005)
plt.tight_layout()
plt.savefig(OUT_DIR / "gradcam_correct.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {OUT_DIR / 'gradcam_correct.png'}")

---
## 9 — Grad-CAM Gallery: Incorrect Predictions (up to 16)

In [ ]:
if len(wrong_df) == 0:
    print("No misclassifications — skipping Grad-CAM incorrect gallery.")
else:
    show_wrong = wrong_df.head(min(len(wrong_df), 16)).reset_index(drop=True)
    n_show     = len(show_wrong)

    fig, axes = plt.subplots(n_show, 2, figsize=(7, n_show * 3.2))
    if n_show == 1:
        axes = axes.reshape(1, 2)

    for i, row in show_wrong.iterrows():
        pil_img  = load_raw_image(row["image_path"])
        tensor   = img_to_tensor(pil_img)
        # Grad-CAM for the PREDICTED (wrong) class — shows what the model saw
        cam_np, _ = gradcam.compute(tensor, class_idx=int(row["pred_idx"]))
        overlay  = make_overlay(pil_img, cam_np)

        true_short = row["true_label"].split("___")[-1].replace("_", " ")
        pred_short = row["pred_label"].split("___")[-1].replace("_", " ")

        axes[i, 0].imshow(pil_img.resize((IMG_SIZE, IMG_SIZE)))
        axes[i, 0].set_title(
            f"TRUE : {true_short[:25]}\nPRED : {pred_short[:25]}\nConf : {row['confidence']:.4f}",
            fontsize=6.5, color="darkred"
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title("Grad-CAM (predicted class)", fontsize=7)
        axes[i, 1].axis("off")

        torch.cuda.empty_cache()

    fig.suptitle(f"Grad-CAM — Incorrect Predictions ({n_show} of {len(wrong_df)} shown)",
                 fontsize=11, y=1.005)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "gradcam_incorrect.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved → {OUT_DIR / 'gradcam_incorrect.png'}")

---
## 10 — Grad-CAM Gallery: Borderline Correct (8 images)
Correct predictions where the model had the lowest confidence — most revealing for understanding model uncertainty.

In [ ]:
borderline_df = correct_df.sort_values("confidence", ascending=True).head(8).reset_index(drop=True)

n_show = len(borderline_df)
fig, axes = plt.subplots(n_show, 2, figsize=(7, n_show * 3.2))
if n_show == 1:
    axes = axes.reshape(1, 2)

for i, row in borderline_df.iterrows():
    pil_img  = load_raw_image(row["image_path"])
    tensor   = img_to_tensor(pil_img)
    cam_np, _ = gradcam.compute(tensor, class_idx=int(row["pred_idx"]))
    overlay  = make_overlay(pil_img, cam_np)

    label_short = row["true_label"].split("___")[-1].replace("_", " ")

    axes[i, 0].imshow(pil_img.resize((IMG_SIZE, IMG_SIZE)))
    axes[i, 0].set_title(
        f"{label_short[:30]}\nConf: {row['confidence']:.4f}  (borderline)",
        fontsize=7, color="darkorange"
    )
    axes[i, 0].axis("off")

    axes[i, 1].imshow(overlay)
    axes[i, 1].set_title("Grad-CAM", fontsize=7)
    axes[i, 1].axis("off")

    torch.cuda.empty_cache()

fig.suptitle("Grad-CAM — Borderline Correct (Lowest-Confidence Correct Predictions)",
             fontsize=11, y=1.005)
plt.tight_layout()
plt.savefig(OUT_DIR / "gradcam_borderline.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {OUT_DIR / 'gradcam_borderline.png'}")

---
## 11 — Interpretation

### Error Analysis Summary

With ~28 misclassifications out of 8,124 test images, the ResNet50 fine-tuned model is operating near its practical ceiling on PlantVillage. The errors are not random — they concentrate in a small number of visually similar class pairs:

- **Corn Cercospora vs. Corn Northern Leaf Blight** (worst pair): Both diseases produce elongated lesions on corn leaves. Cercospora lesions are narrower and grey-brown; NLB lesions are larger and tan. Under certain lighting or image crops, the boundary blurs — this explains the model's F1 of 0.94 for Cercospora (worst in the project).
- **Tomato Spider Mites** (recall=0.98): Spider mite damage appears as fine stippling across the entire leaf surface — a subtle, distributed texture rather than a discrete lesion. A few images where stippling is light or early-stage get misclassified as healthy.
- **Tomato Early Blight / Target Spot** (F1=0.99 each): Both produce concentric-ring lesions. Early Blight rings are yellower; Target Spot rings are darker brown. The model occasionally confuses these at low confidence.

### Grad-CAM Interpretation

**Correct high-confidence predictions**: Grad-CAM activation should concentrate tightly on the primary disease lesion — scab spots, rust pustules, or powdery mildew colonies — with minimal background activation. This confirms the model is learning disease-relevant features, not leaf shape or background soil color.

**Incorrect predictions**: The Grad-CAM for a misclassified image often shows diffuse activation spread across the leaf rather than a sharp lesion focus. This is the "background shortcut" signal — the model activated on texture gradients or lighting patterns that happen to correlate with the wrong class.

**Borderline correct predictions**: These images typically have either (a) multiple lesion types visible, (b) early-stage symptoms that haven't fully developed, or (c) unusual imaging angles. The Grad-CAM heat map for borderlines is often split across two regions — suggesting the model is uncertain between two candidate features.

### Is accuracy a reliable metric here?
Given class sizes ranging from 23 (Potato healthy) to 824 (Orange Haunglongbing), accuracy is skewed toward large classes. Macro-F1 at 99.52% is the honest metric — it weights each of the 38 classes equally regardless of support. The 0.94 F1 on Corn Cercospora (only 77 test samples) drags the macro average down disproportionately.

### Next Steps
- **Phase 9 (Robustness)**: Test macro-F1 degradation under GaussianBlur, JPEG compression (q=30), and 20% random occlusion
- **Phase 10 (FastAPI)**: Wrap ResNet50 in a `/predict` endpoint with preprocessing matching these exact val transforms
- **Phase 11 (ONNX)**: Export to opset 17 and benchmark latency vs PyTorch inference

In [ ]:
# Final summary printout
print("=" * 60)
print("PHASE 7+8 — ERROR ANALYSIS & GRAD-CAM COMPLETE")
print("=" * 60)
print(f"Total test images       : {n_total:,}")
print(f"Correct                 : {n_correct:,} ({100*n_correct/n_total:.2f}%)")
print(f"Misclassified           : {n_wrong:,}  ({100*n_wrong/n_total:.2f}%)")
print()
print("Output files:")
for f in sorted(OUT_DIR.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:6.1f} KB")